# 实践 7.2  高维图像性状的筛选与共线性诊断

《统计遗传与智能育种学》第七章 · 电子资源 `07b_phenomics_feature_qc.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：07b_phenomics_feature_qc.ipynb。实践生成80个基因型、3个重复和120个相互相关的虚拟图像性状，人为加入低重复性特征、近重复特征和批次偏移。代码用单因素随机效应思想计算示例ICC，结合近重复相关和批次偏移规则生成逐性状“保留/警告/删除＋原因”记录表；随后对保留/警告特征做PCA，并用加对角微扰的相关矩阵条件数演示整体冗余。这里没有逐项计算VIF，以免把单一指标当成唯一的筛选规则。

运行检查：重启内核并运行全部单元  
预期：生成每个性状的ICC、近重复相关、批次偏移、处理动作和原因；能识别预设低重复性特征、近重复特征以及至少一个批次偏移；输出PCA解释率与共线性条件数。

## 输入、完整代码、实际输出与解释

输入与可复现条件：80个基因型×3重复×120个模拟图像特征；预设加入10个低重复性特征、近重复特征和批次偏移；SEED=702。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=702
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 702


In [2]:
from sklearn.decomposition import PCA
ng,nrep,nf=80,3,120; latent=rng.normal(size=(ng,10)); loading=rng.normal(size=(10,nf)); base=latent@loading; rows=[]
for r in range(nrep):
 X=base+rng.normal(0,.7,(ng,nf)); X[:,100:110]=rng.normal(0,4,(ng,10)); X[:,110:115]=X[:,:5]+rng.normal(0,.03,(ng,5)); X[:,115:]+= (r==2)*2.5; rows.append(X)
A=np.stack(rows,axis=1); # genotype × repeat × feature
# 单因素随机效应近似ICC：区分基因型间与重复内方差；负估计截断为0
gmean=A.mean(1); ms_between=nrep*np.var(gmean,axis=0,ddof=1); ms_within=np.sum((A-gmean[:,None,:])**2,axis=(0,1))/(ng*(nrep-1)); rep_rel=np.clip((ms_between-ms_within)/(ms_between+(nrep-1)*ms_within+1e-9),0,1)
mean_feat=gmean; C=np.corrcoef(mean_feat,rowvar=False); max_corr=np.max(np.abs(C-np.eye(nf)),axis=1); batch_shift=np.abs(A[:,2,:].mean(0)-A[:,:2,:].mean((0,1)))/(A.reshape(-1,nf).std(0)+1e-9); report=pd.DataFrame({'feature':[f'F{i:03d}' for i in range(nf)],'repeatability_ICC':rep_rel,'max_abs_corr':max_corr,'batch_shift_z':batch_shift})
def classify(row):
 reasons=[]
 if row.repeatability_ICC<.25: reasons.append('low_repeatability')
 if row.max_abs_corr>.995: reasons.append('near_duplicate')
 if row.batch_shift_z>.75: reasons.append('batch_shift')
 action='delete' if 'low_repeatability' in reasons else ('warn' if reasons else 'keep'); return pd.Series([action,';'.join(reasons) if reasons else 'none'])
report[['action','reason']]=report.apply(classify,axis=1); print(report.action.value_counts()); print(report.tail(22).round(3).to_string(index=False))

keep=report.action.isin(['keep','warn']); X=mean_feat[:,keep]; pca=PCA(n_components=5).fit(X); print('PCA explained variance first5',np.round(pca.explained_variance_ratio_,3),'cumulative',round(pca.explained_variance_ratio_.sum(),3))
# 共线性诊断用相关矩阵条件数，避免对120个特征逐个机械算VIF
cond=np.linalg.cond(np.corrcoef(X,rowvar=False)+1e-6*np.eye(X.shape[1])); print('correlation-matrix condition number',round(float(cond),1))


action
keep      98
warn      12
delete    10
Name: count, dtype: int64
feature  repeatability_ICC  max_abs_corr  batch_shift_z action            reason
   F098              0.964         0.785          0.040   keep              none
   F099              0.967         0.821          0.010   keep              none
   F100              0.000         0.204          0.235 delete low_repeatability
   F101              0.153         0.349          0.190 delete low_repeatability
   F102              0.000         0.308          0.125 delete low_repeatability
   F103              0.000         0.244          0.073 delete low_repeatability
   F104              0.000         0.308          0.165 delete low_repeatability
   F105              0.000         0.360          0.115 delete low_repeatability
   F106              0.000         0.326          0.057 delete low_repeatability
   F107              0.000         0.263          0.112 delete low_repeatability
   F108              0.056         0.

结果解释：规则得到98 keep、12 warn、10 delete，预设低重复性F100–F109均被删除，近重复F110–F114被警告，并标记部分批次偏移。PCA前5轴累计解释0.693，加对角微扰后的相关矩阵条件数约1.81×10⁷；原矩阵因秩不足而奇异。ICC和阈值是示例QC规则，正式研究需结合重复设计、测量误差和多重检验。

可修改练习：把近重复相关阈值从0.995改为0.95，或把批次偏移阈值提高，比较审计表动作变化并说明阈值应怎样预注册。